In [1]:
from pyiceberg.catalog import load_catalog
import duckdb

PROJECT = "/Users/Shared/academy/learner/projects/ethereum_iceberg"
catalog = load_catalog(
    "local",
    type="sql",
    uri=f"sqlite:///{PROJECT}/catalog/pyiceberg_catalog.db",
    warehouse=f"file://{PROJECT}/warehouse",
)
S3 = "s3://aws-public-blockchain/v1.0/eth"
con = duckdb.connect()
con.execute("set s3_region = 'us-east-2'")
catalog.list_namespaces()

[('raw',), ('ref',)]

In [2]:
blocks = catalog.load_table("raw.blocks")
scan = blocks.scan(
    row_filter="date >= '2026-10-01'",
    selected_fields=("number", "date", "transaction_count", "base_fee_per_gas"),
)
scan.to_arrow().num_rows

222451

In [3]:
scan.to_pandas().describe()

,number,base_fee_per_gas,transaction_count
count,2.224510e+05,2.224510e+05,222451.000000
mean,2.620496e+07,4.857661e+08,248.886636
std,6.421622e+04,5.317495e+09,106.517101
min,2.609374e+07,2.019197e+07,0.000000
25%,2.614935e+07,6.003936e+07,191.000000
50%,2.620496e+07,9.203329e+07,239.000000
75%,2.626058e+07,1.799981e+08,294.000000
max,2.631619e+07,2.527532e+11,2830.000000


In [4]:
october = scan.to_duckdb(table_name="october")
october.sql("select date, count(*) as blocks, sum(transaction_count) as txs from october group by 1 order by 1 limit 7")

┌────────────┬────────┬─────────┐
│    date    │ blocks │   txs   │
│    date    │ int64  │ int128  │
├────────────┼────────┼─────────┤
│ 2026-10-01 │   7176 │ 2036041 │
│ 2026-10-02 │   7168 │ 1875646 │
│ 2026-10-03 │   7179 │ 1562292 │
│ 2026-10-04 │   7179 │ 1526358 │
│ 2026-10-05 │   7175 │ 1774270 │
│ 2026-10-06 │   7168 │ 1957602 │
│ 2026-10-07 │   7173 │ 1963594 │
└────────────┴────────┴─────────┘